In [ ]:
!pip install -q tokenizers datasets

In [ ]:
%%writefile config.py

from dataclasses import dataclass


@dataclass
class GPTConfig:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.1
    bias: bool = True


@dataclass
class TrainConfig:
    data_dir: str = "data"
    out_dir: str = "checkpoints"
    ckpt_name: str = "ckpt.pt"

    batch_size: int = 64
    grad_accum_steps: int = 1

    max_iters: int = 3000
    eval_interval: int = 250
    eval_iters: int = 50

    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    warmup_iters: int = 200
    lr_decay_iters: int = 3000
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0

    seed: int = 1337
    log_interval: int = 20

Writing config.py


In [ ]:
%%writefile model.py

import math
import torch
import torch.nn as nn
from torch.nn import functional as F

from config import GPTConfig


class CausalSelfAttention(nn.Module):

    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

        # ek hi linear layer se query, key, value teeno nikalte hain (efficient)
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.size()  # batch, sequence length, embedding dim

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        # (B, n_head, T, head_dim) shape me reshape karte hain
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        # PyTorch ka built-in fast (flash) attention, causal masking automatic
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):


    def __init__(self, config: GPTConfig):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):


    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),   # token embedding
            wpe=nn.Embedding(config.block_size, config.n_embd),   # position embedding
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=nn.LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)


        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)

        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def get_num_params(self):
        n_params = sum(p.numel() for p in self.parameters())
        n_params -= self.transformer.wpe.weight.numel()  # position emb usually excluded
        return n_params

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, \
            f"Sequence length {T} block_size {self.config.block_size} se zyada hai"

        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)

        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
        else:

            logits = self.lm_head(x[:, [-1], :])
            loss = None

        return logits, loss

    def configure_optimizer(self, weight_decay, learning_rate, betas):

        decay_params = [p for n, p in self.named_parameters() if p.dim() >= 2 and p.requires_grad]
        nodecay_params = [p for n, p in self.named_parameters() if p.dim() < 2 and p.requires_grad]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, fused=torch.cuda.is_available())
        return optimizer

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

Writing model.py


In [ ]:
%%writefile prepare_data.py

import os
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer

NUM_TRAIN_STORIES = 120_000
NUM_VAL_STORIES = 2_000
VOCAB_SIZE = 8000
DATA_DIR = "data"
# --------------------------------------------------------------------

os.makedirs(DATA_DIR, exist_ok=True)


def main():
    print("Step 1/4: TinyStories dataset download ho raha hai (HuggingFace se)...")
    ds = load_dataset("roneneldan/TinyStories")

    train_texts = ds["train"]["text"][:NUM_TRAIN_STORIES]
    val_texts = ds["validation"]["text"][:NUM_VAL_STORIES]


    train_texts = [t.strip() for t in train_texts if len(t.strip()) > 20]
    val_texts = [t.strip() for t in val_texts if len(t.strip()) > 20]

    print(f"  -> {len(train_texts)} train stories, {len(val_texts)} val stories")


    raw_train_path = os.path.join(DATA_DIR, "raw_train.txt")
    with open(raw_train_path, "w", encoding="utf-8") as f:
        for t in train_texts:
            f.write(t.strip() + "\n<|endofstory|>\n")

    print("Step 2/4: BPE tokenizer train ho raha hai (isi dataset par, chhota vocab)...")
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.train(
        files=[raw_train_path],
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=["<|endofstory|>"],
    )
    tokenizer.save_model(DATA_DIR)
    print(f"  -> Tokenizer saved to {DATA_DIR}/ (vocab size = {tokenizer.get_vocab_size()})")

    eos_id = tokenizer.token_to_id("<|endofstory|>")

    def encode_texts(texts):
        ids = []
        for t in tqdm(texts, desc="tokenizing"):
            enc = tokenizer.encode(t.strip())
            ids.extend(enc.ids)
            ids.append(eos_id)
        return ids

    print("Step 3/4: Train data tokenize ho raha hai...")
    train_ids = encode_texts(train_texts)
    print("Step 4/4: Validation data tokenize ho raha hai...")
    val_ids = encode_texts(val_texts)

    train_ids = np.array(train_ids, dtype=np.uint16)
    val_ids = np.array(val_ids, dtype=np.uint16)

    train_ids.tofile(os.path.join(DATA_DIR, "train.bin"))
    val_ids.tofile(os.path.join(DATA_DIR, "val.bin"))

    print(f"\nDone! train tokens = {len(train_ids):,} | val tokens = {len(val_ids):,}")
    print(f"Actual vocab size = {tokenizer.get_vocab_size()}  <-- yeh config.py me vocab_size me daalo agar 8000 se different ho")


if __name__ == "__main__":
    main()

Writing prepare_data.py


In [ ]:
%%writefile train.py

import os
import math
import time
import json
import numpy as np
import torch

from config import GPTConfig, TrainConfig
from model import GPT

train_cfg = TrainConfig()
model_cfg = GPTConfig()

torch.manual_seed(train_cfg.seed)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

use_amp = device == "cuda"
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and amp_dtype == torch.float16))

vocab_path = os.path.join(train_cfg.data_dir, "vocab.json")
if os.path.exists(vocab_path):
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab = json.load(f)
    model_cfg.vocab_size = len(vocab)
print(f"vocab_size = {model_cfg.vocab_size}")


def get_batch(split):
    path = os.path.join(train_cfg.data_dir, f"{split}.bin")
    data = np.memmap(path, dtype=np.uint16, mode="r")
    ix = torch.randint(len(data) - model_cfg.block_size - 1, (train_cfg.batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i + model_cfg.block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + 1 + model_cfg.block_size].astype(np.int64)) for i in ix])
    if device == "cuda":
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(train_cfg.eval_iters)
        for k in range(train_cfg.eval_iters):
            X, Y = get_batch(split)
            with torch.autocast(device_type=device if device != "mps" else "cpu", dtype=amp_dtype, enabled=use_amp):
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def get_lr(it):
    if it < train_cfg.warmup_iters:
        return train_cfg.learning_rate * (it + 1) / train_cfg.warmup_iters
    if it > train_cfg.lr_decay_iters:
        return train_cfg.min_lr
    decay_ratio = (it - train_cfg.warmup_iters) / (train_cfg.lr_decay_iters - train_cfg.warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return train_cfg.min_lr + coeff * (train_cfg.learning_rate - train_cfg.min_lr)


def main():
    os.makedirs(train_cfg.out_dir, exist_ok=True)

    model = GPT(model_cfg).to(device)
    print(f"Model params: {model.get_num_params() / 1e6:.2f}M")

    optimizer = model.configure_optimizer(
        train_cfg.weight_decay, train_cfg.learning_rate, (train_cfg.beta1, train_cfg.beta2)
    )

    if device == "cuda":
        try:
            model = torch.compile(model)
            print("torch.compile enabled (fast)")
        except Exception as e:
            print("torch.compile skip kiya:", e)

    best_val_loss = float("inf")
    t0 = time.time()

    for it in range(train_cfg.max_iters + 1):
        lr = get_lr(it)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        if it % train_cfg.eval_interval == 0 or it == train_cfg.max_iters:
            losses = estimate_loss(model)
            elapsed = (time.time() - t0) / 60
            print(f"step {it}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, "
                  f"elapsed {elapsed:.1f} min")

            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
                torch.save({
                    "model_state_dict": raw_model.state_dict(),
                    "model_config": model_cfg,
                    "iter": it,
                    "val_loss": best_val_loss,
                }, os.path.join(train_cfg.out_dir, train_cfg.ckpt_name))
                print(f"  -> checkpoint saved (val loss {best_val_loss:.4f})")

        X, Y = get_batch("train")
        with torch.autocast(device_type=device if device != "mps" else "cpu", dtype=amp_dtype, enabled=use_amp):
            _, loss = model(X, Y)
            loss = loss / train_cfg.grad_accum_steps

        scaler.scale(loss).backward()

        if (it + 1) % train_cfg.grad_accum_steps == 0:
            if train_cfg.grad_clip != 0.0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        if it % train_cfg.log_interval == 0:
            print(f"  iter {it}: loss {loss.item() * train_cfg.grad_accum_steps:.4f}, lr {lr:.6f}")

    total_min = (time.time() - t0) / 60
    print(f"\nTraining complete in {total_min:.1f} minutes. Best val loss: {best_val_loss:.4f}")


if __name__ == "__main__":
    main()

Writing train.py


In [ ]:
%%writefile generate.py

import os
import argparse
import torch
from tokenizers import ByteLevelBPETokenizer

from model import GPT
from config import TrainConfig

train_cfg = TrainConfig()


def load_model(ckpt_path, device):
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model_cfg = checkpoint["model_config"]
    model = GPT(model_cfg)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    print(f"Model loaded (trained till iter {checkpoint['iter']}, val loss {checkpoint['val_loss']:.4f})")
    return model


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", type=str, default="Once upon a time")
    parser.add_argument("--max_new_tokens", type=int, default=200)
    parser.add_argument("--temperature", type=float, default=0.8)
    parser.add_argument("--top_k", type=int, default=50)
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = ByteLevelBPETokenizer(
        os.path.join(train_cfg.data_dir, "vocab.json"),
        os.path.join(train_cfg.data_dir, "merges.txt"),
    )

    ckpt_path = os.path.join(train_cfg.out_dir, train_cfg.ckpt_name)
    model = load_model(ckpt_path, device)

    ids = tokenizer.encode(args.prompt).ids
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        y = model.generate(x, args.max_new_tokens, temperature=args.temperature, top_k=args.top_k)

    out_ids = y[0].tolist()
    text = tokenizer.decode(out_ids)
    text = text.split("<|endofstory|>")[0]  # pehli story tak hi rakho

    print("\n----- GENERATED TEXT -----\n")
    print(text)
    print("\n---------------------------\n")


if __name__ == "__main__":
    main()

Writing generate.py


In [ ]:
!python prepare_data.py

Step 1/4: TinyStories dataset download ho raha hai (HuggingFace se)...
README.md: 100% 1.06k/1.06k [00:00<00:00, 572kB/s]
data/train-00000-of-00004-2d5a1467fff108(…): 100% 249M/249M [00:02<00:00, 86.9MB/s]
data/train-00001-of-00004-5852b56a2bd28f(…): 100% 248M/248M [00:02<00:00, 84.7MB/s]
data/train-00002-of-00004-a26307300439e9(…): 100% 246M/246M [00:02<00:00, 84.0MB/s]
data/train-00003-of-00004-d243063613e5a0(…): 100% 248M/248M [00:02<00:00, 91.8MB/s]
data/validation-00000-of-00001-869c898b5(…): 100% 9.99M/9.99M [00:00<00:00, 12.9MB/s]
Generating train split: 100% 2119719/2119719 [00:10<00:00, 207759.31 examples/s]
Generating validation split: 100% 21990/21990 [00:00<00:00, 143618.18 examples/s]
  -> 119972 train stories, 2000 val stories
Step 2/4: BPE tokenizer train ho raha hai (isi dataset par, chhota vocab)...
[00:00:00] Tokenize words                 ██████████████████ 28149    /    28149
[00:00:00] Count pairs                    ██████████████████ 28149    /    28149
[00:00:00]

In [ ]:
!python train.py

Using device: cuda
vocab_size = 8000
Model params: 13.72M
torch.compile enabled (fast)
W0715 18:19:43.188000 4135 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2941: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
step 0: train loss 9.0816, val loss 9.0784, elapsed 0.6 min
  -> checkpoint saved (val loss 9.0784)
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2941: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2941: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2941: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
  iter 0: loss 9

In [ ]:
!python generate.py --prompt "what is your name" --max_new_tokens 100

Model loaded (trained till iter 3000, val loss 2.1939)

----- GENERATED TEXT -----

what is your name?"

Mia looks at Tom. She sees Tom's name. She feels sorry. She says, "I'm sorry, Tom. I forgive you. But I don't want to be your friend. I will be your friend."

Tom looks at Mia. He sees it. He says, "Can I play with you?"

Mia says, "Yes, you can. You are not my friend. You are my friend. You are my friend."

Tom says

---------------------------

